# Backends: what this build dispatches to

One mtl5 call can run MTL5's own C++ kernels or a vendor's BLAS and LAPACK.
Which one is decided when mtl5 is compiled, not when it runs. So the questions
are what this build has, and what it costs to have it or not.

**Uses:** `mtl5.build_info`, `mtl5.system_info`, `mtl5.backends`,
`mtl5.get_backend`, `mtl5.set_backend`, `mtl5.matmul`.

**From C++:** [`phase09_advanced_types/blas_dispatch.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase09_advanced_types/blas_dispatch.cpp).

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

import mtl5

# Chart styling: recessive axes and grid, ink for text, validated categorical hues.
SURFACE = "#fcfcfb"
INK, INK_2 = "#0b0b0b", "#52514e"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)

## What was compiled in

`build_info()` reports the compile-time switches of this build, and
`system_info()` reports the machine it runs on. The C++ example prints its
`MTL5_HAS_BLAS` and `MTL5_HAS_LAPACK` macros; these dictionaries are the same
facts at run time.

In [ ]:
build, system = mtl5.build_info(), mtl5.system_info()
for key in ("blas", "lapack", "native_fast_gemm", "highway_simd", "kpu", "build_isa"):
    print(f"{key:>17}: {build[key]}")
print(f"{'machine':>17}: {system['cpu_brand']}, SIMD {system['cpu_simd']}")

`build_isa` is the instruction set the binary was compiled for, and
`cpu_simd` is what the machine supports. A portable wheel targets the
baseline of its platform, so it runs on every machine of that architecture and
uses no instruction a newer CPU adds. A build from source with
`-C cmake.define.MTL5_NATIVE_ARCH=ON` targets the machine it is built on.

## The dispatch order

`backends()` lists the backends of this build in dispatch order, and
`get_backend()` names the one dense kernels use: the first available of KPU,
BLAS, native and reference.

In [ ]:
available = mtl5.backends()
print("available:", available, " in use:", mtl5.get_backend())
assert mtl5.get_backend() == available[0]

`set_backend` can't switch to a backend this build doesn't have, because
the choice was made by the compiler. It confirms a backend is present, and if
it isn't, the error says how to rebuild.

In [ ]:
mtl5.set_backend(available[0])  # present: no error

missing = [name for name in ("kpu", "blas", "native", "reference") if name not in available]
for name in missing:
    try:
        mtl5.set_backend(name)
    except RuntimeError as err:
        print(f"{name}: {str(err).split('. ')[0]}.")
    else:
        raise AssertionError(f"set_backend({name!r}) should raise in this build")

## What it costs: GEMM against NumPy's BLAS

NumPy links a BLAS of its own, usually OpenBLAS, so `a @ b` is an
optimized vendor GEMM on any machine. Timing `mtl5.matmul` against it on the
same matrices shows where this build stands. The C++
example makes the same comparison against a textbook triple loop.

The results must agree to rounding, whatever the speed: the difference is
checked against the forward-error bound for a dot product of length n,
n · ε · |A| |B|.

In [ ]:
def best_of(f, repeats=5):
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        f()
        times.append(time.perf_counter() - start)
    return min(times)


rng = np.random.default_rng(7)
sizes = [64, 128, 256, 512]
rates = {"mtl5.matmul": [], "NumPy @ (its own BLAS)": []}
for n in sizes:
    a, b = rng.standard_normal((n, n)), rng.standard_normal((n, n))
    A, B = mtl5.matrix(a), mtl5.matrix(b)

    c_mtl5, c_numpy = mtl5.matmul(A, B).to_numpy(), a @ b
    bound = n * np.finfo(float).eps * (np.abs(a) @ np.abs(b))
    assert (np.abs(c_mtl5 - c_numpy) <= bound).all(), n

    flops = 2.0 * n**3
    rates["mtl5.matmul"].append(flops / best_of(lambda: mtl5.matmul(A, B)) / 1e9)
    rates["NumPy @ (its own BLAS)"].append(flops / best_of(lambda: a @ b) / 1e9)

fig, ax = plt.subplots(figsize=(5.8, 3.6))
for color, (label, r) in zip(SERIES, rates.items()):
    ax.plot(sizes, r, "o-", color=color, label=label)
ax.set_xscale("log", base=2)
ax.set_xticks(sizes, [str(n) for n in sizes])
ax.set_xlabel("n (n × n times n × n, float64)")
ax.set_ylabel("GFLOP/s")
ax.set_title(
    f"GEMM on {system['cpu_brand']}\n"
    f"mtl5: {mtl5.get_num_threads()} thread(s), built for {build['build_isa']}; "
    "NumPy: its own BLAS",
    fontsize=9,
)
ax.legend(frameon=False)
ax.grid(True)
plt.show()
print("all products agree with NumPy's within n·ε·|A||B|")

These timings come from the machine that built this page, and they vary from
run to run, so nothing above asserts them. Only the agreement is checked.

Two things make the gap. NumPy's OpenBLAS runs threads of its own, while
mtl5 runs one thread unless `mtl5.set_num_threads(n)` is called before its first
operation. And OpenBLAS picks its kernels for the CPU it finds at run time
(`np.show_config()` shows how it was built), while a portable mtl5 wheel was
compiled for its platform's baseline (`build_isa` above), whatever the CPU
supports.

To have mtl5 call a vendor BLAS and LAPACK, or target the machine's own
instruction set, build from source:

```bash
pip install . -C cmake.define.MTL5_WITH_BLAS=ON -C cmake.define.MTL5_WITH_LAPACK=ON
pip install . -C cmake.define.MTL5_NATIVE_ARCH=ON   # this machine only
```

## From C++

| MTL5 (C++) | mtl5 (Python) |
| --- | --- |
| `#ifdef MTL5_HAS_BLAS`, `#ifdef MTL5_HAS_LAPACK` | `mtl5.build_info()["blas"]`, `["lapack"]` |
| `cmake -DMTL5_ENABLE_BLAS=ON -DMTL5_ENABLE_LAPACK=ON` | `pip install . -C cmake.define.MTL5_WITH_BLAS=ON -C cmake.define.MTL5_WITH_LAPACK=ON` |
| `interface::blas::gemm(...)` behind `mult(A, B, C)` | `mtl5.matmul(A, B)`, which dispatches to `mtl5.get_backend()` |
| the generic C++ GEMM | the `native` backend |